# 🤖 Agente de IA para Análise de Feedback de Alunos
**Google Colab + Flask + ngrok + n8n + WhatsApp (Twilio)**

---
Execute as células **em ordem**, de cima para baixo.

## Célula 1 — Instalar bibliotecas

In [ ]:
!pip install transformers torch flask pyngrok --quiet
print('✅ Bibliotecas instaladas!')

✅ Bibliotecas instaladas!


## Célula 2 — Carregar modelo e definir funções

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Modelo multilíngue de sentimento (5 classes: 1 a 5 estrelas)
model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

def analisar_sentimento(texto: str):
    """
    Recebe um texto e retorna:
    - texto original
    - estrelas (1 a 5)
    - sentimento (positivo, negativo, neutro)
    """
    inputs = tokenizer(texto, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        scores = torch.nn.functional.softmax(outputs.logits, dim=1)[0]

    estrelas = torch.argmax(scores).item() + 1  # 0-4 -> 1-5

    if estrelas in [1, 2]:
        sentimento = "negativo"
    elif estrelas == 3:
        sentimento = "neutro"
    else:
        sentimento = "positivo"

    return {
        "texto": texto,
        "estrelas": int(estrelas),
        "sentimento": sentimento
    }

def decidir_acao(resultado_sentimento):
    """
    Heurística:
    - Se sentimento negativo e <= 2 estrelas: ALERTA
    - Se sentimento positivo e >= 4 estrelas: ELOGIO
    - Caso contrário: NEUTRO
    """
    sentimento = resultado_sentimento["sentimento"]
    estrelas = resultado_sentimento["estrelas"]

    if sentimento == "negativo" and estrelas <= 2:
        return "ALERTA"
    elif sentimento == "positivo" and estrelas >= 4:
        return "ELOGIO"
    else:
        return "NEUTRO"

print('✅ Modelo carregado e funções definidas!')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Modelo carregado e funções definidas!


## Célula 3 — Testar o modelo localmente

In [ ]:
frases_teste = [
    "Gostei muito da aula de hoje, foi bem explicada.",
    "Achei a aula muito confusa e difícil de entender.",
    "A aula foi ok, nada demais."
]

for f in frases_teste:
    r = analisar_sentimento(f)
    acao = decidir_acao(r)
    print("Frase:", f)
    print("Resultado:", r)
    print("Ação (heurística):", acao)
    print("-" * 60)

Frase: Gostei muito da aula de hoje, foi bem explicada.
Resultado: {'texto': 'Gostei muito da aula de hoje, foi bem explicada.', 'estrelas': 4, 'sentimento': 'positivo'}
Ação (heurística): ELOGIO
------------------------------------------------------------
Frase: Achei a aula muito confusa e difícil de entender.
Resultado: {'texto': 'Achei a aula muito confusa e difícil de entender.', 'estrelas': 2, 'sentimento': 'negativo'}
Ação (heurística): ALERTA
------------------------------------------------------------
Frase: A aula foi ok, nada demais.
Resultado: {'texto': 'A aula foi ok, nada demais.', 'estrelas': 3, 'sentimento': 'neutro'}
Ação (heurística): NEUTRO
------------------------------------------------------------


## 🖊️ Exercício 1 — Teste suas próprias frases
Substitua as frases abaixo e execute a célula.

In [ ]:
minhas_frases = [
    "Aula foi muito legal, o conteudo foi bem explicado",
    "Não consegui entender a aula, foi muito rapido a aula hoje",
    "Nada novo na aula hoje, tudo ok"
]

for f in minhas_frases:
    r = analisar_sentimento(f)
    acao = decidir_acao(r)
    print("Frase:", f)
    print("Resultado:", r)
    print("Ação (heurística):", acao)
    print("-" * 60)

Frase: Aula foi muito legal, o conteudo foi bem explicado
Resultado: {'texto': 'Aula foi muito legal, o conteudo foi bem explicado', 'estrelas': 4, 'sentimento': 'positivo'}
Ação (heurística): ELOGIO
------------------------------------------------------------
Frase: Não consegui entender a aula, foi muito rapido a aula hoje
Resultado: {'texto': 'Não consegui entender a aula, foi muito rapido a aula hoje', 'estrelas': 2, 'sentimento': 'negativo'}
Ação (heurística): ALERTA
------------------------------------------------------------
Frase: Nada novo na aula hoje, tudo ok
Resultado: {'texto': 'Nada novo na aula hoje, tudo ok', 'estrelas': 3, 'sentimento': 'neutro'}
Ação (heurística): NEUTRO
------------------------------------------------------------


## Célula 4 — Configurar ngrok
⚠️ **Substitua `COLOQUE_SEU_TOKEN_AQUI` pelo seu Auth Token do ngrok** (https://ngrok.com)

In [ ]:
from pyngrok import ngrok

# ⚠️ COLOQUE SEU TOKEN AQUI ENTRE ASPAS
NGROK_AUTH_TOKEN = "SEU TOKEN AQUI"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(5000)

print("🌐 URL pública do servidor Flask:", public_url)
print()
print("📌 Anote essa URL! Você vai usá-la no n8n.")
print("   Exemplo de uso: " + str(public_url) + "/sentimento")

🌐 URL pública do servidor Flask: NgrokTunnel: "https://exfoliate-careless-gap.ngrok-free.dev" -> "http://localhost:5000"

📌 Anote essa URL! Você vai usá-la no n8n.
   Exemplo de uso: NgrokTunnel: "https://exfoliate-careless-gap.ngrok-free.dev" -> "http://localhost:5000"/sentimento


## Célula 5 — Criar e iniciar a API Flask
⚠️ **Deixe essa célula rodando.** Se interrompê-la, o servidor cai.

In [ ]:
from flask import Flask, request, jsonify
import threading

app = Flask(__name__)

@app.route("/sentimento", methods=["POST"])
def sentimento_route():
    """
    Espera JSON: { "texto": "A aula foi muito confusa." }
    Retorna:     { "texto": ..., "sentimento": ..., "estrelas": ..., "acao": ... }
    """
    data = request.get_json(force=True)
    texto = data.get("texto", "")

    resultado = analisar_sentimento(texto)
    acao = decidir_acao(resultado)

    resposta = {
        "texto": resultado["texto"],
        "sentimento": resultado["sentimento"],
        "estrelas": resultado["estrelas"],
        "acao": acao
    }
    return jsonify(resposta)

def run_app():
    app.run(host="0.0.0.0", port=5000)

thread = threading.Thread(target=run_app)
thread.start()

print("✅ Servidor Flask rodando na porta 5000!")
print("   Rota disponível: POST /sentimento")

✅ Servidor Flask rodando na porta 5000!
   Rota disponível: POST /sentimento


## Célula 6 — Testar a API

In [ ]:
import requests

# ✅ Extrai só a URL limpa do objeto ngrok
base_url = public_url.public_url
url = base_url + "/sentimento"

print("🧪 Testando:", url)

resp = requests.post(url, json={"texto": "A aula foi muito confusa e cansativa."})

print("Status code:", resp.status_code)
print("Resposta JSON:", resp.json())

🧪 Testando: https://exfoliate-careless-gap.ngrok-free.dev/sentimento


INFO:werkzeug:127.0.0.1 - - [15/Apr/2026 03:42:23] "POST /sentimento HTTP/1.1" 200 -


Status code: 200
Resposta JSON: {'acao': 'ALERTA', 'estrelas': 2, 'sentimento': 'negativo', 'texto': 'A aula foi muito confusa e cansativa.'}
